# Voice Cloning + Talking Head Generation (Free & Open Source)

This notebook guides you through:
1. Cloning your voice from a reference audio sample.
2. Generating a talking-head video from your photo and the cloned audio.

All tools used are free and open source:
- **Voice cloning**: Tortoise‑TTS (latent conditioning) or Coqui TTS (YourTTS).
- **Video generation**: Wav2Lip (lip‑sync) or SadTalker (more expressive).

📌 **Prerequisites**:
- A clear reference audio of your voice (≥10 s, low noise, wav format preferred).
- A frontal photo of your face (shoulders‑up, well‑lit).
- A Google account (to run Colab).

⏱️ **Expected runtime**:
- Voice cloning: 1‑5 min (depends on length).
- Video generation: ~0.5‑2 s per frame on GPU (Tesla T4 in Colab).
  For a 10‑second clip at 25 fps expect ~4‑8 min.

---


In [ ]:
# 1️⃣ Install system dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg libsm6 libxext6  > /dev/null 2>&1
print('System dependencies installed.')


In [ ]:
# 2️⃣ Install Python packages
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q git+https://github.com/neonbjb/tortoise-tts.git@main
!pip install -q TTS  # Coqui TTS
!pip install -q opencv-python librosa soundfile tqdm
print('Python packages installed.')


## 3️⃣ Upload your reference audio and photo

Run the cell below and select:
- `reference.wav` (or .mp3, .flac) – your voice sample.
- `photo.jpg` (or .png) – your portrait.


In [ ]:
from google.colab import files
uploaded = files.upload()
for fn in uploaded.keys():
  print(f'Uploaded: {fn}')


## 4️⃣ Prepare files
Rename uploaded files to expected names if needed.

In [ ]:
import os
for f in os.listdir('.'):
    if f.lower().endswith(('.wav', '.mp3', '.flac')) and 'reference' not in f.lower():
        os.rename(f, 'reference.wav')
        print(f'Renamed {f} -> reference.wav')
    if f.lower().endswith(('.jpg', '.jpeg', '.png')) and 'photo' not in f.lower():
        os.rename(f, 'photo.jpg')
        print(f'Renamed {f} -> photo.jpg')


## 5️⃣ Voice Cloning with Tortoise-TTS
This method uses latent conditioning from a reference audio to clone voice.
It works well with short reference audio (a few seconds).
We'll generate audio for a given text.


In [ ]:
# Text you want the cloned voice to say
text = "Hello, this is a test of voice cloning and talking head generation."
print(f'Text to synthesize: {text}')


In [ ]:
from tortoise.utils.audio import load_audio, load_voice
from tortoise.api import TextToSpeech
from tortoise.utils.text import split_and_recombine_text
import torch
tts = TextToSpeech()

# Load reference audio to get voice conditioning
voice_samples, conditioning_latents = load_voice('reference.wav', extra_voice_paths=None)
print('Voice samples shape:', voice_samples.shape)
print('Conditioning latents shape:', conditioning_latents.shape)


In [ ]:
# Generate audio
gen = tts.tts_with_preset(
    text,
    voice_samples=voice_samples,
    conditioning_latents=conditioning_latents,
    preset='fast',  # options: 'ultra_fast', 'fast', 'standard', 'high_quality'
    use_deterministic_seed=True
)
torchaudio.save('cloned_voice.wav', gen.squeeze(0).cpu(), 24000)
print('Saved cloned_voice.wav')


## 6️⃣ (Alternative) Voice Cloning with Coqui TTS (YourTTS)
If you prefer Coqui TTS, you can run the following instead.
Comment out the Tortoise section above and run this block.


In [ ]:
# Uncomment to run Coqui TTS
# from TTS.api import TTS
# tts = TTS(model_name="tts_models/multilingual/multi-dataset/yourtts", progress_bar=False, gpu=True)
# tts.tts_to_file(text=text, speaker_wav="reference.wav", language="en", file_path="cloned_voice_coqui.wav")
# print('Saved cloned_voice_coqui.wav')


## 7️⃣ Download Wav2Lip model
We'll use the pre‑trained Wav2Lip model (requires face detection).


In [ ]:
!mkdir -p checkpoints
!if [ ! -f checkpoints/wav2lip.pth ]; then \
    wget https://github.com/Rudrabha/Wav2Lip/releases/download/v1.0/wav2lip.pth -O checkpoints/wav2lip.pth; \
fi
print('Wav2Lip model ready.')


## 8️⃣ Run Wav2Lip inference
This will produce a lip‑synced video using the cloned audio and your photo.
We'll use the generic face detector included in the repo.


In [ ]:
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip
!python inference.py \
  --checkpoint_path ../checkpoints/wav2lip.pth \
  --face ../photo.jpg \
  --audio ../cloned_voice.wav \
  --outfile ../result_voicecloned.wav2lip.mp4
%cd ..
print('Video generated: result_voicecloned.wav2lip.mp4')


## 9️⃣ Download results
Run the cell below to download the generated video(s) to your local machine.


In [ ]:
from google.colab import files
if os.path.exists('result_voicecloned.wav2lip.mp4'):
    files.download('result_voicecloned.wav2lip.mp4')
    print('Downloaded Wav2Lip result.')
else:
    print('Video not found yet; maybe inference still running.')


---
### Tips for longer videos (3‑10 min)
- Generate a longer audio script (e.g., read a passage, repeat sentences, or use a text‑to‑speech script that produces several minutes of speech).
- Split the audio into chunks of ~10‑20 seconds, run Wav2Lip on each chunk, then concatenate with `ffmpeg -f concat -safe 0 -i <(for f in ./chunks/*.mp4; do echo "file '$PWD/$f'"; done) -c copy final.mp4`.
- For voice cloning, Tortoise‑TTS can handle longer texts but may become slower; Coqui TTS is often faster for longer synthesis.
- Ensure your reference audio is clean and contains varied phonemes for better cloning.
- If you run into GPU memory limits, reduce the batch size or use the `fast` preset in Tortoise.

Enjoy creating your AI avatar!
